In [0]:
# CELL 1
from pyspark.sql import functions as F

df = spark.read.table("sarkarimitracatalog.sarkaridatabase.schemes_structured")

df_priority = df.withColumn(
    "priority_score",
    F.when(F.col("level") == "central", 3).otherwise(1) +
    F.when(F.col("income_limit_inr") < 100000, 2).otherwise(0) +
    F.when(F.col("extraction_confidence") > 0.85, 1).otherwise(0) +
    F.when(F.col("disability_required") == True, 1).otherwise(0)
).withColumn(
    "urgency_label",
    F.when(F.col("priority_score") >= 5, "critical")
     .when(F.col("priority_score") >= 3, "high")
     .when(F.col("priority_score") >= 2, "medium")
     .otherwise("standard")
).select("slug", "scheme_name", "priority_score", "urgency_label",
         "level", "schemeCategory")

print("✅ Priority distribution:")
display(df_priority.groupBy("urgency_label").count())

✅ Priority distribution:


urgency_label,count
medium,2487
high,502
standard,369
critical,41


In [0]:
# CELL 2 — Write
df_priority.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sarkarimitracatalog.sarkaridatabase.scheme_priority_rules")

print("✅ Written: sarkarimitracatalog.sarkaridatabase.scheme_priority_rules")

✅ Written: sarkarimitracatalog.sarkaridatabase.scheme_priority_rules
